# Preparação dos indicadores RIPSA para o Power BI

Este notebook implementa a primeira camada descrita em `direcionamento.md`: monitoramento de **casos registrados** por tempo e território.

A base atual não sustenta incidência, risco, mortalidade, letalidade, cobertura vacinal, completude ou classificação final dos casos. Também não é usada para Machine Learning nesta etapa, pois o direcionamento define previsão e classificação de risco como evoluções futuras dependentes de dados adicionais.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

ARQUIVO_ENTRADA = Path('ripsa008mb.csv')
DIRETORIO_SAIDA = Path('saidas_powerbi')
DIRETORIO_SAIDA.mkdir(exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda valor: f'{valor:,.4f}')

## 1. Carga e validação do esquema

Os códigos territoriais são carregados como texto para preservar zeros à esquerda e evitar relações incorretas no Power BI.

In [ ]:
COLUNAS_CODIGO = [
    'co_anomes', 'co_ibge', 'co_uf', 'co_regiao_brasil',
    'co_regiao_saude', 'co_macro', 'sg_categoria', 'co_item_categoria'
]

COLUNAS_OBRIGATORIAS = {
    'co_anomes', 'co_ibge', 'vl_indicador_calculado_mun',
    'co_uf', 'no_municipio', 'sg_uf', 'no_uf',
    'co_regiao_brasil', 'no_regiao_brasil', 'sg_regiao_brasil',
    'co_regiao_saude', 'no_regiao_saude', 'co_macro', 'no_macro',
    'sg_categoria', 'ds_categoria', 'co_item_categoria', 'ds_item_categoria'
}

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {ARQUIVO_ENTRADA.resolve()}')

dados = pd.read_csv(
    ARQUIVO_ENTRADA,
    dtype={coluna: 'string' for coluna in COLUNAS_CODIGO},
    low_memory=False,
)

colunas_ausentes = sorted(COLUNAS_OBRIGATORIAS.difference(dados.columns))
if colunas_ausentes:
    raise ValueError(f'Colunas obrigatórias ausentes: {colunas_ausentes}')

dados['vl_indicador_calculado_mun'] = pd.to_numeric(
    dados['vl_indicador_calculado_mun'], errors='raise'
)

if not dados['co_anomes'].str.fullmatch(r'\d{6}').all():
    invalidos = dados.loc[~dados['co_anomes'].str.fullmatch(r'\d{6}'), 'co_anomes'].head().tolist()
    raise ValueError(f'Valores inválidos em co_anomes: {invalidos}')

dados['data_referencia'] = pd.to_datetime(dados['co_anomes'], format='%Y%m', errors='raise')
dados['ano'] = dados['data_referencia'].dt.year.astype('int16')
dados['mes'] = dados['data_referencia'].dt.month.astype('int8')

print(f'Registros carregados: {len(dados):,}')
print(f'Período: {dados["ano"].min()} a {dados["ano"].max()}')
dados.head()

## 2. Validação temporal e prevenção de dupla contagem

A inspeção factual mostra que todos os registros estão posicionados em dezembro. Portanto, `co_anomes` representa aqui uma referência **anual**, e não uma série mensal completa. O notebook não produz sazonalidade nem comparações mensais.

Para o total oficial, somente `sg_categoria = "TC"` (Todas as categorias) pode ser somada. As outras categorias repetem o mesmo total em diferentes desagregações.

In [ ]:
meses_disponiveis = sorted(dados['mes'].unique().tolist())
if meses_disponiveis == [12]:
    warnings.warn(
        'Todos os registros estão em dezembro. Análises mensais e de sazonalidade '
        'não são sustentadas por esta base; serão geradas apenas séries anuais.',
        stacklevel=1,
    )
else:
    raise ValueError(
        'A cobertura temporal mudou. Revise a granularidade antes de habilitar análises mensais: '
        f'meses encontrados = {meses_disponiveis}'
    )

totais_por_categoria = (
    dados.groupby(['sg_categoria', 'ds_categoria'], dropna=False, as_index=False)
    .agg(registros=('co_ibge', 'size'), total=('vl_indicador_calculado_mun', 'sum'))
    .sort_values('sg_categoria')
)
display(totais_por_categoria)

fato = dados.loc[dados['sg_categoria'].eq('TC')].copy()
fato['teve_caso'] = fato['vl_indicador_calculado_mun'].gt(0).astype('int8')
fato['municipio_ano_com_ocorrencia'] = fato['teve_caso']

assert len(fato) == 72_410, f'Esperados 72.410 registros TC; encontrados {len(fato):,}.'
assert not fato.duplicated(['co_anomes', 'co_ibge']).any(), 'A chave município-ano não é única.'
assert set(fato['ano'].unique()) == set(range(2012, 2025)), 'O período esperado é 2012 a 2024.'

municipios_por_ano_validacao = fato.groupby('ano')['co_ibge'].nunique()
assert municipios_por_ano_validacao.eq(5_570).all(), (
    'Cada ano deve conter 5.570 municípios. Divergências: '
    f'{municipios_por_ano_validacao[~municipios_por_ano_validacao.eq(5_570)].to_dict()}'
)

total_casos_validado = fato['vl_indicador_calculado_mun'].sum()
assert np.isclose(total_casos_validado, 41_020), (
    f'O total TC esperado é 41.020; encontrado {total_casos_validado:,.0f}.'
)

fato_faixa_etaria = dados.loc[
    dados['sg_categoria'].eq('FXETC1')
    & dados['co_item_categoria'].notna()
    & dados['ds_item_categoria'].notna()
].copy()
fato_faixa_etaria['co_item_categoria'] = (
    fato_faixa_etaria['co_item_categoria'].str.replace(r'\.0$', '', regex=True)
)

ORDEM_FAIXA_ETARIA = {
    'Menor de 1 ano': 1,
    '1 a 4 anos': 2,
    '5 a 9 anos': 3,
    '10 a 14 anos': 4,
    '15 a 19 anos': 5,
    '20 a 29 anos': 6,
    '30 a 39 anos': 7,
    '40 a 49 anos': 8,
    '50 a 59 anos': 9,
    '60 anos ou mais': 10,
}
fato_faixa_etaria['ordem_faixa_etaria'] = (
    fato_faixa_etaria['ds_item_categoria'].map(ORDEM_FAIXA_ETARIA)
)

assert fato_faixa_etaria['ordem_faixa_etaria'].notna().all(), (
    'Foram encontradas faixas etárias sem ordem definida.'
)
assert not fato_faixa_etaria.duplicated([
    'co_anomes', 'co_ibge', 'co_item_categoria'
]).any(), 'A chave município-ano-faixa etária não é única.'
assert np.isclose(
    fato_faixa_etaria['vl_indicador_calculado_mun'].sum(), total_casos_validado
), 'O total por faixa etária diverge do total TC.'
totais_etarios_anuais = fato_faixa_etaria.groupby('ano')['vl_indicador_calculado_mun'].sum()
totais_tc_anuais = fato.groupby('ano')['vl_indicador_calculado_mun'].sum()
pd.testing.assert_series_equal(
    totais_etarios_anuais, totais_tc_anuais, check_names=False
)

print(f'Registros TC: {len(fato):,}')
print(f'Registros com faixa etária: {len(fato_faixa_etaria):,}')
print(f'Total de casos registrado e validado: {total_casos_validado:,.0f}')
print(f'Municípios por ano: {municipios_por_ano_validacao.min():,}')

## 3. Tabela fato e calendário anual

A tabela fato possui uma linha por município e ano. Ela é a fonte auditável dos totais e pode ser relacionada ao calendário anual e às tabelas-resumo.

In [ ]:
COLUNAS_FATO = [
    'data_referencia', 'co_anomes', 'ano', 'mes',
    'co_ibge', 'no_municipio', 'co_uf', 'sg_uf', 'no_uf',
    'co_regiao_brasil', 'sg_regiao_brasil', 'no_regiao_brasil',
    'co_regiao_saude', 'no_regiao_saude', 'co_macro', 'no_macro',
    'sg_categoria', 'ds_categoria', 'vl_indicador_calculado_mun',
    'teve_caso', 'municipio_ano_com_ocorrencia'
]

fato_powerbi = fato[COLUNAS_FATO].sort_values(['ano', 'co_ibge']).reset_index(drop=True)

COLUNAS_FATO_FAIXA_ETARIA = [
    'data_referencia', 'co_anomes', 'ano', 'mes',
    'co_ibge', 'no_municipio', 'co_uf', 'sg_uf', 'no_uf',
    'co_regiao_brasil', 'sg_regiao_brasil', 'no_regiao_brasil',
    'co_regiao_saude', 'no_regiao_saude', 'co_macro', 'no_macro',
    'sg_categoria', 'ds_categoria', 'co_item_categoria',
    'ds_item_categoria', 'ordem_faixa_etaria', 'vl_indicador_calculado_mun'
]
fato_faixa_etaria_powerbi = (
    fato_faixa_etaria[COLUNAS_FATO_FAIXA_ETARIA]
    .sort_values(['ano', 'co_ibge', 'ordem_faixa_etaria'])
    .reset_index(drop=True)
)

dim_faixa_etaria = (
    fato_faixa_etaria_powerbi[[
        'co_item_categoria', 'ds_item_categoria', 'ordem_faixa_etaria'
    ]]
    .drop_duplicates()
    .sort_values('ordem_faixa_etaria')
    .reset_index(drop=True)
)

dim_calendario_anual = pd.DataFrame({'ano': range(2012, 2025)})
dim_calendario_anual['data_ano'] = pd.to_datetime(
    dim_calendario_anual['ano'].astype(str) + '-01-01'
)
dim_calendario_anual['ano_anterior'] = dim_calendario_anual['ano'] - 1

fato_powerbi.head()

## 4. Resumos anual, territorial, concentração e persistência

In [ ]:
resumo_anual = (
    fato.groupby('ano', as_index=False)
    .agg(
        casos_registrados=('vl_indicador_calculado_mun', 'sum'),
        municipios_analisados=('co_ibge', 'nunique'),
        municipios_com_ocorrencia=('teve_caso', 'sum'),
        media_casos_por_municipio=('vl_indicador_calculado_mun', 'mean'),
        maximo_casos_em_um_municipio=('vl_indicador_calculado_mun', 'max'),
    )
    .sort_values('ano')
)
resumo_anual['municipios_sem_ocorrencia'] = (
    resumo_anual['municipios_analisados'] - resumo_anual['municipios_com_ocorrencia']
)

ufs_com_ocorrencia = (
    fato.loc[fato['teve_caso'].eq(1)]
    .groupby('ano')['sg_uf'].nunique()
    .rename('ufs_com_ocorrencia')
)
resumo_anual = resumo_anual.merge(ufs_com_ocorrencia, on='ano', how='left')
resumo_anual['ufs_com_ocorrencia'] = resumo_anual['ufs_com_ocorrencia'].fillna(0).astype(int)
resumo_anual['variacao_anual_percentual'] = (
    resumo_anual['casos_registrados'].pct_change(fill_method=None).mul(100)
)

municipios_por_ano = fato_powerbi[[
    'ano', 'co_ibge', 'no_municipio', 'sg_uf', 'no_regiao_brasil',
    'vl_indicador_calculado_mun', 'teve_caso'
]].rename(columns={'vl_indicador_calculado_mun': 'casos_registrados'})

resumo_faixa_etaria = (
    fato_faixa_etaria.groupby([
        'co_item_categoria', 'ds_item_categoria', 'ordem_faixa_etaria'
    ], as_index=False)
    .agg(
        casos_registrados=('vl_indicador_calculado_mun', 'sum'),
        municipios_com_ocorrencia=('co_ibge', 'nunique'),
        anos_com_ocorrencia=('ano', 'nunique'),
    )
    .sort_values('ordem_faixa_etaria')
)
resumo_faixa_etaria['participacao_percentual'] = (
    resumo_faixa_etaria['casos_registrados'].div(total_casos_validado).mul(100)
)
resumo_faixa_etaria['ranking_casos'] = (
    resumo_faixa_etaria['casos_registrados'].rank(method='min', ascending=False).astype(int)
)

resumo_faixa_etaria_anual = (
    fato_faixa_etaria.groupby([
        'ano', 'co_item_categoria', 'ds_item_categoria', 'ordem_faixa_etaria'
    ], as_index=False)
    .agg(
        casos_registrados=('vl_indicador_calculado_mun', 'sum'),
        municipios_com_ocorrencia=('co_ibge', 'nunique'),
    )
    .sort_values(['ano', 'ordem_faixa_etaria'])
)
resumo_faixa_etaria_anual['participacao_no_ano_percentual'] = (
    resumo_faixa_etaria_anual['casos_registrados']
    .div(resumo_faixa_etaria_anual.groupby('ano')['casos_registrados'].transform('sum'))
    .mul(100)
)

display(resumo_anual)

In [ ]:
def primeiro_ano_com_ocorrencia(grupo):
    anos = grupo.loc[grupo['vl_indicador_calculado_mun'].gt(0), 'ano']
    return int(anos.min()) if not anos.empty else pd.NA


def ultimo_ano_com_ocorrencia(grupo):
    anos = grupo.loc[grupo['vl_indicador_calculado_mun'].gt(0), 'ano']
    return int(anos.max()) if not anos.empty else pd.NA


def classificar_persistencia(anos_com_ocorrencia):
    if anos_com_ocorrencia == 0:
        return 'Sem ocorrência'
    if anos_com_ocorrencia == 1:
        return 'Ocorrência isolada'
    if anos_com_ocorrencia <= 4:
        return 'Recorrente'
    return 'Persistente'


dimensoes_municipio = [
    'co_ibge', 'no_municipio', 'co_uf', 'sg_uf', 'no_uf',
    'co_regiao_brasil', 'no_regiao_brasil',
    'co_regiao_saude', 'no_regiao_saude', 'co_macro', 'no_macro'
]

resumo_municipio = (
    fato.groupby(dimensoes_municipio, dropna=False, as_index=False)
    .agg(
        casos_registrados=('vl_indicador_calculado_mun', 'sum'),
        anos_analisados=('ano', 'nunique'),
        anos_com_ocorrencia=('teve_caso', 'sum'),
        maior_volume_anual=('vl_indicador_calculado_mun', 'max'),
    )
)

limites_ocorrencia = (
    fato.groupby('co_ibge', sort=False)
    .apply(
        lambda grupo: pd.Series({
            'primeiro_ano_com_ocorrencia': primeiro_ano_com_ocorrencia(grupo),
            'ultimo_ano_com_ocorrencia': ultimo_ano_com_ocorrencia(grupo),
        }),
        include_groups=False,
    )
    .reset_index()
)
resumo_municipio = resumo_municipio.merge(limites_ocorrencia, on='co_ibge', how='left')
resumo_municipio['classificacao_persistencia'] = (
    resumo_municipio['anos_com_ocorrencia'].map(classificar_persistencia)
)

total_geral = resumo_municipio['casos_registrados'].sum()
resumo_municipio['participacao_percentual'] = np.where(
    total_geral > 0,
    resumo_municipio['casos_registrados'].div(total_geral).mul(100),
    0.0,
)
resumo_municipio['ranking_casos'] = (
    resumo_municipio['casos_registrados'].rank(method='min', ascending=False).astype(int)
)
resumo_municipio = resumo_municipio.sort_values(
    ['casos_registrados', 'co_ibge'], ascending=[False, True]
).reset_index(drop=True)
resumo_municipio['participacao_acumulada_percentual'] = (
    resumo_municipio['participacao_percentual'].cumsum()
)

resumo_municipio.head(10)

In [ ]:
def criar_resumo_territorial(colunas_grupo):
    resumo = (
        fato.groupby(colunas_grupo, dropna=False, as_index=False)
        .agg(
            casos_registrados=('vl_indicador_calculado_mun', 'sum'),
            municipios_analisados=('co_ibge', 'nunique'),
        )
    )
    ocorrencias = (
        fato.loc[fato['teve_caso'].eq(1)]
        .groupby(colunas_grupo, dropna=False)
        .agg(
            municipios_com_ocorrencia=('co_ibge', 'nunique'),
            anos_com_ocorrencia=('ano', 'nunique'),
        )
        .reset_index()
    )
    resumo = resumo.merge(ocorrencias, on=colunas_grupo, how='left')
    resumo[['municipios_com_ocorrencia', 'anos_com_ocorrencia']] = (
        resumo[['municipios_com_ocorrencia', 'anos_com_ocorrencia']].fillna(0).astype(int)
    )
    resumo['participacao_percentual'] = np.where(
        total_geral > 0, resumo['casos_registrados'].div(total_geral).mul(100), 0.0
    )
    resumo['ranking_casos'] = resumo['casos_registrados'].rank(
        method='min', ascending=False
    ).astype(int)
    return resumo.sort_values(
        ['casos_registrados'] + colunas_grupo,
        ascending=[False] + [True] * len(colunas_grupo),
    ).reset_index(drop=True)


resumo_uf = criar_resumo_territorial(['co_uf', 'sg_uf', 'no_uf'])
resumo_regiao = criar_resumo_territorial([
    'co_regiao_brasil', 'sg_regiao_brasil', 'no_regiao_brasil'
])
resumo_regiao_saude = criar_resumo_territorial([
    'co_regiao_saude', 'no_regiao_saude', 'sg_uf'
])
resumo_macrorregiao = criar_resumo_territorial([
    'co_macro', 'no_macro', 'sg_uf'
])

resumo_uf.head(10)

## 5. Resumo geral e exportação

In [ ]:
linha_ano_pico = resumo_anual.loc[resumo_anual['casos_registrados'].idxmax()]
linha_municipio_pico = resumo_municipio.iloc[0]

resumo_geral = pd.DataFrame([{
    'total_casos_registrados': int(total_geral),
    'municipios_analisados': int(fato['co_ibge'].nunique()),
    'municipios_com_ocorrencia': int(fato.loc[fato['teve_caso'].eq(1), 'co_ibge'].nunique()),
    'municipios_sem_ocorrencia': int(
        fato['co_ibge'].nunique() - fato.loc[fato['teve_caso'].eq(1), 'co_ibge'].nunique()
    ),
    'ufs_com_ocorrencia': int(fato.loc[fato['teve_caso'].eq(1), 'sg_uf'].nunique()),
    'ano_maior_volume': int(linha_ano_pico['ano']),
    'casos_no_ano_maior_volume': int(linha_ano_pico['casos_registrados']),
    'codigo_municipio_maior_volume': linha_municipio_pico['co_ibge'],
    'municipio_maior_volume': linha_municipio_pico['no_municipio'],
    'uf_municipio_maior_volume': linha_municipio_pico['sg_uf'],
    'casos_municipio_maior_volume': int(linha_municipio_pico['casos_registrados']),
    'periodo_inicial': int(fato['ano'].min()),
    'periodo_final': int(fato['ano'].max()),
    'granularidade_temporal': 'Anual',
}])

TABELAS = {
    'fato_casos_registrados.csv': fato_powerbi,
    'fato_casos_faixa_etaria.csv': fato_faixa_etaria_powerbi,
    'dim_calendario_anual.csv': dim_calendario_anual,
    'dim_faixa_etaria.csv': dim_faixa_etaria,
    'resumo_geral.csv': resumo_geral,
    'resumo_anual.csv': resumo_anual,
    'municipios_por_ano.csv': municipios_por_ano,
    'resumo_municipio.csv': resumo_municipio,
    'resumo_uf.csv': resumo_uf,
    'resumo_regiao.csv': resumo_regiao,
    'resumo_regiao_saude.csv': resumo_regiao_saude,
    'resumo_macrorregiao.csv': resumo_macrorregiao,
    'resumo_faixa_etaria.csv': resumo_faixa_etaria,
    'resumo_faixa_etaria_anual.csv': resumo_faixa_etaria_anual,
}

catalogo_dados = pd.DataFrame([
    ('fato_casos_registrados.csv', 'Município-ano', 'Fonte auditável dos casos; apenas categoria TC.'),
    ('fato_casos_faixa_etaria.csv', 'Município-ano-faixa etária', 'Casos desagregados por faixa etária; não somar com a fato TC.'),
    ('dim_calendario_anual.csv', 'Ano', 'Dimensão de tempo anual.'),
    ('dim_faixa_etaria.csv', 'Faixa etária', 'Códigos, descrições e ordem de exibição das faixas etárias.'),
    ('resumo_geral.csv', 'Painel', 'KPIs gerais da visão executiva.'),
    ('resumo_anual.csv', 'Ano', 'Evolução, variação anual e extensão territorial.'),
    ('municipios_por_ano.csv', 'Município-ano', 'Mapa e evolução anual dos municípios afetados.'),
    ('resumo_municipio.csv', 'Município', 'Ranking, concentração e persistência.'),
    ('resumo_uf.csv', 'UF', 'Ranking e participação por UF.'),
    ('resumo_regiao.csv', 'Região do Brasil', 'Distribuição por grande região.'),
    ('resumo_regiao_saude.csv', 'Região de Saúde', 'Distribuição por Região de Saúde.'),
    ('resumo_macrorregiao.csv', 'Macrorregião de Saúde', 'Distribuição por macrorregião.'),
    ('resumo_faixa_etaria.csv', 'Faixa etária', 'Distribuição total, participação e ranking por faixa etária.'),
    ('resumo_faixa_etaria_anual.csv', 'Ano-faixa etária', 'Evolução anual e participação de cada faixa etária.'),
], columns=['arquivo', 'grao', 'uso_recomendado'])
TABELAS['catalogo_dados.csv'] = catalogo_dados

for nome_arquivo, tabela in TABELAS.items():
    caminho = DIRETORIO_SAIDA / nome_arquivo
    tabela.to_csv(caminho, index=False, encoding='utf-8-sig')
    print(f'{nome_arquivo}: {len(tabela):,} linhas')

## 6. Validação dos arquivos exportados

In [ ]:
arquivos_ausentes = [
    nome for nome in TABELAS if not (DIRETORIO_SAIDA / nome).exists()
]
assert not arquivos_ausentes, f'Arquivos não exportados: {arquivos_ausentes}'

fato_exportada = pd.read_csv(
    DIRETORIO_SAIDA / 'fato_casos_registrados.csv',
    dtype={'co_anomes': 'string', 'co_ibge': 'string'},
)
assert len(fato_exportada) == 72_410, 'A fato exportada perdeu ou duplicou registros.'
assert np.isclose(fato_exportada['vl_indicador_calculado_mun'].sum(), 41_020), (
    'A fato exportada não totaliza 41.020 casos.'
)
assert fato_exportada['sg_categoria'].eq('TC').all(), 'A fato contém categoria diferente de TC.'
assert not fato_exportada.duplicated(['co_anomes', 'co_ibge']).any(), (
    'A fato exportada não mantém a chave município-ano.'
)

fato_etaria_exportada = pd.read_csv(
    DIRETORIO_SAIDA / 'fato_casos_faixa_etaria.csv',
    dtype={
        'co_anomes': 'string', 'co_ibge': 'string', 'co_item_categoria': 'string'
    },
)
assert len(fato_etaria_exportada) == 3_614, (
    'A fato etária exportada perdeu ou duplicou registros.'
)
assert np.isclose(
    fato_etaria_exportada['vl_indicador_calculado_mun'].sum(), 41_020
), 'A fato etária exportada não totaliza 41.020 casos.'
assert fato_etaria_exportada['sg_categoria'].eq('FXETC1').all(), (
    'A fato etária contém categoria diferente de FXETC1.'
)
assert not fato_etaria_exportada.duplicated([
    'co_anomes', 'co_ibge', 'co_item_categoria'
]).any(), 'A fato etária não mantém a chave município-ano-faixa etária.'
assert fato_etaria_exportada['ds_item_categoria'].nunique() == 10, (
    'A fato etária não contém as 10 faixas esperadas.'
)

print(f'Validação concluída: {len(TABELAS)} arquivos em {DIRETORIO_SAIDA.resolve()}')
display(resumo_geral)

## 7. Medidas DAX sugeridas

A tabela fato já contém exclusivamente a categoria `TC`, mas a medida abaixo mantém o filtro explícito para rastreabilidade.

```DAX
Total de Casos =
CALCULATE(
    SUM(fato_casos_registrados[vl_indicador_calculado_mun]),
    fato_casos_registrados[sg_categoria] = "TC"
)

Municípios com Ocorrência =
CALCULATE(
    DISTINCTCOUNT(fato_casos_registrados[co_ibge]),
    fato_casos_registrados[teve_caso] = 1
)

Municípios Analisados =
DISTINCTCOUNT(fato_casos_registrados[co_ibge])

Proporção de Municípios com Ocorrência =
DIVIDE([Municípios com Ocorrência], [Municípios Analisados])

Participação no Total =
DIVIDE(
    [Total de Casos],
    CALCULATE([Total de Casos], ALLSELECTED(fato_casos_registrados[co_ibge]))
)

Variação Anual =
VAR AnoAtual = SELECTEDVALUE(dim_calendario_anual[ano])
VAR CasosAnoAnterior =
    CALCULATE(
        [Total de Casos],
        FILTER(ALL(dim_calendario_anual), dim_calendario_anual[ano] = AnoAtual - 1)
    )
RETURN
    DIVIDE([Total de Casos] - CasosAnoAnterior, CasosAnoAnterior)
```

> A proporção de municípios com ocorrência não é incidência. Não crie medidas mensais ou de sazonalidade com esta versão da base, pois todos os registros estão associados a dezembro.